### Sketch some stuff

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import FeatureMapping, HierarchicalMuData, make_synthetic_proteomics_data

### Focus on getting the indices back from the adjacency matrix in varp

1. Generate an adjacency matrix and populate it with values

In [2]:
sides = ["GENE1", "GENE2", "PROTEIN1-1", "PROTEIN1-2", "PROTEIN2", "PRECURSOR1", "PRECURSOR2", "PRECURSOR3", "PRECURSOR4"]
am_df = pd.DataFrame({side: [0]*len(sides) for side in sides}, index=sides)

# gene1 maps to protein1-1 and protein1-2
am_df.loc["GENE1", "PROTEIN1-1"] = 1
am_df.loc["GENE1", "PROTEIN1-2"] = 1

# protein1-1 maps to precursor1 and precursor2
am_df.loc["PROTEIN1-1", "PRECURSOR1"] = 1
am_df.loc["PROTEIN1-1", "PRECURSOR2"] = 1

# protein1-2 maps to precursor3
am_df.loc["PROTEIN1-2", "PRECURSOR3"] = 1

# gene2 maps to protein2
am_df.loc["GENE2", "PROTEIN2"] = 1

# protein2 maps to precursor4
am_df.loc["PROTEIN2", "PRECURSOR4"] = 1

# mirror that whole thing to make the matrix symmetric
am_df = am_df + am_df.T 

# every side maps to itself
for side in sides:
    am_df.loc[side, side] = 1

# Define the feature boundaries (we would get this from the constructor)
matrix_bounds = {"GENE": (0, 2), "PROTEIN": (2, 5), "PRECURSOR": (5, 9)}

## Validation

In [3]:
# Check that the level is in the matrix bounds
def _validate_query_level(query_level: str, matrix_bounds: dict[str, tuple[int, int]]):
    if query_level not in matrix_bounds.keys():
        raise ValueError(f"Query level {query_level} not found in matrix bounds.")

# Check that the feature boundaries are consistent with the adjacency matrix size
def _validate_feature_bounds(feature_boundaries: dict[str, tuple[int, int]], adjacency_matrix: np.ndarray):
    total_range = sum(end - start for start, end in feature_boundaries.values())
    if total_range != adjacency_matrix.shape[0]:
        raise ValueError(f"Total range of feature boundaries ({total_range}) does not match adjacency matrix size ({adjacency_matrix.shape[0]}).")

# Check that the query feature is unique in the features lookup
def _check_unique_feature(features: np.ndarray, query: str):
    query_indices = np.where(features == query)[0]
    if len(query_indices) == 0:
        raise ValueError(f"Query '{query}' not found in features.")
    elif len(query_indices) > 1:
        raise ValueError(f"Query '{query}' found multiple times in features at indices {query_indices}.")

## Index translation

In [4]:
def feature_index_to_adjacency_index(
    query_feature_index: int,
    feature_level: str,
    feature_bounds: dict[str, tuple[int, int]],
) -> int:
    """Return actual index of query in adjacency matrix."""
    # Find the start of the requested feature level
    if feature_level not in feature_bounds:
        raise ValueError(f"Feature level '{feature_level}' not found in feature bounds")
    
    start, end = feature_bounds[feature_level]
    
    level_size = end - start
    if query_feature_index < 0 or query_feature_index >= level_size:
        raise ValueError(f"Query index {query_feature_index} out of range for level '{feature_level}' (size: {level_size})")
    
    return start + query_feature_index

def adjacency_index_to_feature_index(
    adjacency_index: int,
    feature_bounds: dict[str, tuple[int, int]],
) -> tuple[str, int]:
    """Return feature level and index of query in adjacency matrix."""
    for level, (start, end) in feature_bounds.items():
        if start <= adjacency_index < end:
            return level, adjacency_index - start
    raise ValueError(f"Adjacency index {adjacency_index} out of bounds for feature bounds.")

## Slicing operation

Assumes getting a unique numerical feature index for the current modality. Workflow would be

Loading precursor / protein / gene AnnData objects --> each of them has numerical feature indices --> creating an adjacency matrix from the concatenated indices + feature bounds

In [5]:
def slice_associated_features(
    query_feature_index: int,
    feature_level: str,
    feature_bounds: dict[str, tuple[int, int]],
    adjacency_matrix: np.ndarray,
) -> dict[str, list[int]]:
    """Convert a query feature index into a map of associated features across all levels.
    
    Returns:
        Dictionary mapping feature level names to lists of feature indices that are
        connected to the query feature in the adjacency matrix.
    """
    
    # Convert query feature index to adjacency matrix index
    query_adjacency_index = feature_index_to_adjacency_index(
        query_feature_index=query_feature_index,
        feature_level=feature_level,
        feature_bounds=feature_bounds,
    )

    # Slice adjacency matrix to get associated features (where value is 1)
    associated_adjacency_indices = np.where(adjacency_matrix[query_adjacency_index, :] == 1)[0]
    
    # Convert adjacency indices back to feature indices grouped by level
    associated_features = {level: [] for level in feature_bounds.keys()}
    
    for adj_idx in associated_adjacency_indices:
        level, feature_idx = adjacency_index_to_feature_index(
            adjacency_index=adj_idx,
            feature_bounds=feature_bounds,
        )
        associated_features[level].append(feature_idx)
    
    return associated_features

In [6]:
slice_associated_features(
    query_feature_index=0,
    feature_level="PROTEIN",
    feature_bounds=matrix_bounds,
    adjacency_matrix=am_df.values
)

{'GENE': [np.int64(0)],
 'PROTEIN': [np.int64(0)],
 'PRECURSOR': [np.int64(0), np.int64(1)]}

In [7]:
# Test: get the second protein (PROTEIN1-2) which is at index 1 in the protein level
# PROTEIN level starts at index 2 in the adjacency matrix, so PROTEIN1-2 should be at index 3
adj_idx = feature_index_to_adjacency_index(
    query_feature_index=1,
    feature_level="PROTEIN",
    feature_bounds=matrix_bounds,
)
print(f"PROTEIN[1] maps to adjacency index: {adj_idx}")
print(f"This corresponds to: {sides[adj_idx]}")

PROTEIN[1] maps to adjacency index: 3
This corresponds to: PROTEIN1-2


In [8]:
# Comprehensive test: Query different features and show their associations
print("="*60)
print("Testing hierarchical feature mapping")
print("="*60)

# Test 1: Query GENE1 (index 0 in GENE level)
print("\n1. Query GENE1 (GENE[0]):")
result = slice_associated_features(
    query_feature_index=0,
    feature_level="GENE",
    feature_bounds=matrix_bounds,
    adjacency_matrix=am_df.values
)
print(f"Associated features: {result}")
print(f"  - Self: GENE[{result['GENE']}] = {[sides[i] for i in result['GENE']]}")
print(f"  - Proteins: PROTEIN[{result['PROTEIN']}] = {[sides[i+2] for i in result['PROTEIN']]}")
print(f"  - Precursors: PRECURSOR[{result['PRECURSOR']}] = {[sides[i+5] for i in result['PRECURSOR']]}")

# Test 2: Query PROTEIN1-2 (index 1 in PROTEIN level)  
print("\n2. Query PROTEIN1-2 (PROTEIN[1]):")
result = slice_associated_features(
    query_feature_index=1,
    feature_level="PROTEIN",
    feature_bounds=matrix_bounds,
    adjacency_matrix=am_df.values
)
print(f"Associated features: {result}")
print(f"  - Genes: GENE[{result['GENE']}] = {[sides[i] for i in result['GENE']]}")
print(f"  - Self: PROTEIN[{result['PROTEIN']}] = {[sides[i+2] for i in result['PROTEIN']]}")
print(f"  - Precursors: PRECURSOR[{result['PRECURSOR']}] = {[sides[i+5] for i in result['PRECURSOR']]}")

# Test 3: Query PRECURSOR2 (index 1 in PRECURSOR level)
print("\n3. Query PRECURSOR2 (PRECURSOR[1]):")
result = slice_associated_features(
    query_feature_index=1,
    feature_level="PRECURSOR",
    feature_bounds=matrix_bounds,
    adjacency_matrix=am_df.values
)
print(f"Associated features: {result}")
print(f"  - Genes: GENE[{result['GENE']}] = {[sides[i] for i in result['GENE']]}")
print(f"  - Proteins: PROTEIN[{result['PROTEIN']}] = {[sides[i+2] for i in result['PROTEIN']]}")
print(f"  - Self: PRECURSOR[{result['PRECURSOR']}] = {[sides[i+5] for i in result['PRECURSOR']]}")

Testing hierarchical feature mapping

1. Query GENE1 (GENE[0]):
Associated features: {'GENE': [np.int64(0)], 'PROTEIN': [np.int64(0), np.int64(1)], 'PRECURSOR': []}
  - Self: GENE[[np.int64(0)]] = ['GENE1']
  - Proteins: PROTEIN[[np.int64(0), np.int64(1)]] = ['PROTEIN1-1', 'PROTEIN1-2']
  - Precursors: PRECURSOR[[]] = []

2. Query PROTEIN1-2 (PROTEIN[1]):
Associated features: {'GENE': [np.int64(0)], 'PROTEIN': [np.int64(1)], 'PRECURSOR': [np.int64(2)]}
  - Genes: GENE[[np.int64(0)]] = ['GENE1']
  - Self: PROTEIN[[np.int64(1)]] = ['PROTEIN1-2']
  - Precursors: PRECURSOR[[np.int64(2)]] = ['PRECURSOR3']

3. Query PRECURSOR2 (PRECURSOR[1]):
Associated features: {'GENE': [], 'PROTEIN': [np.int64(0)], 'PRECURSOR': [np.int64(1)]}
  - Genes: GENE[[]] = []
  - Proteins: PROTEIN[[np.int64(0)]] = ['PROTEIN1-1']
  - Self: PRECURSOR[[np.int64(1)]] = ['PRECURSOR2']


In [9]:
### LOAD HERE - Try with backed mode to avoid loading varp
import mudata as md
import warnings

test_data_path = "/Users/vincenthbrennsteiner/Documents/mann_labs/_git_repositories/202603_hackathon_proteomics/msmudata/data/minimal1.h5mu"

# Try loading in backed mode first to see structure
try:
    mdata = md.read(test_data_path, backed='r')
    print("Loaded in backed mode:")
    print(mdata)
except Exception as e:
    print(f"Backed mode failed: {e}")
    
# If that doesn't work, try regular loading with suppressed warnings
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mdata = md.read(test_data_path)
        print("\nLoaded with warnings suppressed:")
        print(mdata)
except Exception as e:
    print(f"\nRegular loading failed: {e}")

Loaded in backed mode:
MuData object with n_obs × n_vars = 5 × 11 backed at '/Users/vincenthbrennsteiner/Documents/mann_labs/_git_repositories/202603_hackathon_proteomics/msmudata/data/minimal1.h5mu'
  varp:	'feature_mapping'
  3 modalities
    precursors:	5 x 6
      var:	'genes', 'proteins', 'precursors'
    proteins:	5 x 3
      var:	'proteins', 'genes'
    genes:	5 x 2

Loaded with warnings suppressed:
MuData object with n_obs × n_vars = 5 × 11
  varp:	'feature_mapping'
  3 modalities
    precursors:	5 x 6
      var:	'genes', 'proteins', 'precursors'
    proteins:	5 x 3
      var:	'proteins', 'genes'
    genes:	5 x 2


/Users/vincenthbrennsteiner/miniconda3/envs/msmudata/lib/python3.11/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/Users/vincenthbrennsteiner/miniconda3/envs/msmudata/lib/python3.11/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [10]:
import mudata as md
from scipy import sparse
import warnings

# Suppress the FutureWarning for now since pull_on_update=False seems to cause issues
with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    
    # Load the MuData
    test_data_path = "/Users/vincenthbrennsteiner/Documents/mann_labs/_git_repositories/202603_hackathon_proteomics/msmudata/data/minimal1.h5mu"
    mdata = md.read(test_data_path)

print("MuData structure:")
print(mdata)
print("\nModalities:", list(mdata.mod.keys()))

MuData structure:
MuData object with n_obs × n_vars = 5 × 11
  varp:	'feature_mapping'
  3 modalities
    precursors:	5 x 6
      var:	'genes', 'proteins', 'precursors'
    proteins:	5 x 3
      var:	'proteins', 'genes'
    genes:	5 x 2

Modalities: ['precursors', 'proteins', 'genes']


In [11]:
# Explore the structure of each modality
for mod_name, adata in mdata.mod.items():
    print(f"\n{mod_name.upper()} modality:")
    print(f"  Shape: {adata.shape}")
    print(f"  Observations (samples): {adata.n_obs}")
    print(f"  Variables (features): {adata.n_vars}")
    if hasattr(adata, 'var') and not adata.var.empty:
        print(f"  Variable columns: {list(adata.var.columns)}")
        print(f"  First few features:")
        print(adata.var.head(3))


PRECURSORS modality:
  Shape: (5, 6)
  Observations (samples): 5
  Variables (features): 6
  Variable columns: ['genes', 'proteins', 'precursors']
  First few features:
   genes  proteins precursors
1  gene0  protein0          1
2  gene1  protein1          2
3  gene1  protein2          3

PROTEINS modality:
  Shape: (5, 3)
  Observations (samples): 5
  Variables (features): 3
  Variable columns: ['proteins', 'genes']
  First few features:
          proteins  genes
protein0  protein0  gene0
protein1  protein1  gene1
protein2  protein2  gene1

GENES modality:
  Shape: (5, 2)
  Observations (samples): 5
  Variables (features): 2


## Actual example dataset

From https://github.com/orgs/scverse/projects/71/views/1?pane=issue&itemId=170549166&issue=scverse%7C202603_hackathon_proteomics%7C7

In [12]:
### LOAD HERE

import mudata as md

test_data_path = "/Users/vincenthbrennsteiner/Documents/mann_labs/_git_repositories/202603_hackathon_proteomics/msmudata/data/minimal1.h5mu"
mdata = md.read(test_data_path)

print(mdata)

MuData object with n_obs × n_vars = 5 × 11
  varp:	'feature_mapping'
  3 modalities
    precursors:	5 x 6
      var:	'genes', 'proteins', 'precursors'
    proteins:	5 x 3
      var:	'proteins', 'genes'
    genes:	5 x 2


/Users/vincenthbrennsteiner/miniconda3/envs/msmudata/lib/python3.11/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/Users/vincenthbrennsteiner/miniconda3/envs/msmudata/lib/python3.11/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
